## Low-Rank Adaptation (LoRA)

**LoRA** is a popular parameter-efficient fine-tuning (PEFT) technique used to adapt large pre-trained models (like GPT-4, Llama 3) to specific tasks or styles without the massive computational cost of retraining the entire model.

### 1. The Problem: Full Fine-Tuning
Traditional fine-tuning requires updating **every single parameter** (weight) in the neural network.

* **Memory Constraints:** For a model with billions of parameters, this requires massive amounts of GPU memory (VRAM) to store optimizer states and gradients (often 3-4x the model size).
* **Storage Issues:** After training, the new model is just as large as the original (e.g., 100GB), making it difficult to store or share multiple versions.

**Mathematical Context:**
If $W_{pre}$ is the pre-trained weight and $W_{ft}$ is the fine-tuned weight, we are trying to learn the full difference matrix $\Delta W$:
$$W_{pre} → W_{ft} $$
$$W_{ft} - W_{pre} = \Delta W$$

### 2. The Solution: Low-Rank Decomposition
LoRA proposes that the change in weights ($\Delta W$) is "low-rank" which means it doesn't need high complexity (full dimension) to be effective.

Instead of updating the massive original matrix $W$, LoRA freezes it and adds a parallel branch of two tiny matrices, **$A$** and **$B$**.

**How it works:**
1.  **Freeze** the pre-trained weight matrix $W$ (it remains untouched).
2.  **Inject** two trainable low-rank matrices, $A$ and $B$, into the layer.
$$
B \in \mathbb{R}^{n_{out} \times r}, \quad A \in \mathbb{R}^{r \times n_{in}}
$$
$$
r \ll n_{in}, \quad r \ll n_{out}
$$

* **$r$ (Rank):** A hyperparameter (e.g., 8, 16, 64) that determines the size of $A$ and $B$. A lower rank means fewer parameters to train.

3.  **Compute** the output ($h$) by adding the original path and the LoRA path:

$$h = W_0 x + \Delta W x$$
$$h = W_0 x + B A x$$

* **Result:** You only train $A$ and $B$. The original $W$ remains untouched.

### 3. Key Benefits
* **Hardware Efficiency:** Reduces the number of trainable parameters by up to 10,000x. You can fine-tune huge models on consumer hardware (e.g., a single NVIDIA RTX 3090 or 4090).
* **Lower Memory Cost:** Significantly decreases VRAM usage during training (e.g., avoids storing optimizer states for the full model).
* **Compact Storage:** LoRA weights are tiny. A LoRA file might be **10MB – 200MB**, whereas a full model checkpoint is **20GB – 100GB**.
* **Modularity:** You can keep one "Base Model" frozen and swap different LoRAs in and out on the fly. (e.g., Switch from a "Coding LoRA" to a "Poetry LoRA" instantly, without needing to host 2 full models).
* **Zero Inference Latency:** If you merge the LoRA weights back into the base model ($W_{new} = W + BA$), the inference speed is identical to the original model.

## Installation

Before running the cells below, please complete the following setup steps to ensure the environment is configured correctly.

### 1. Configure the Runtime (Critical)
You must switch to a GPU runtime for this notebook to work.
* Go to the top menu and select **Runtime** $\rightarrow$ **Change runtime type**.
* Under **Hardware accelerator**, select **T4 GPU**.
* Click **Save**.

### 2. Set API Keys
You need to store your Hugging Face token securely.
* Click on the **Keys** icon in the left sidebar of Google Colab.
* Create a new secret named `HF_TOKEN`.
* Paste your Hugging Face Access Token as the value.
* Toggle the **Notebook access** switch to **"On"** so the notebook can read it.

### 3. Weights & Biases (Optional)
This notebook uses **Unsloth**, which integrates with **Weights & Biases (WandB)** for experiment tracking.
* During the training step, you may be prompted to log in to WandB.
* If you have an account, you can link it when prompted.
* If not, you can choose to skip this step or create a free account to track your loss curves.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes xformers
!pip install openai
!pip install fastapi nest-asyncio pyngrok uvicorn
!pip install vllm

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-u4kxye66/unsloth_7f66f0c3aeb049f8a329e5b0b74bfcfc
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-u4kxye66/unsloth_7f66f0c3aeb049f8a329e5b0b74bfcfc
  Resolved https://github.com/unslothai/unsloth.git to commit b96a04c17bc6bcb5522eafb17adc2b104be38f99
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 91.7 MB/s eta 0:00:00


## Training LoRA Adaptor

This script demonstrates how to efficiently fine-tune the **Llama 3 8B** model using the `unsloth` library and Low-Rank Adaptation (LoRA). It is optimized for free-tier GPUs (like Google Colab T4).

#### Key Steps performed in the code:
1.  **Load Model:** Initializes a 4-bit quantized version of `llama-3-8b` using `FastLanguageModel` for memory efficiency.
- [Link to Model: unsloth/llama-3-8b-bnb-4bit](https://huggingface.co/unsloth/llama-3-8b-bnb-4bit)
2.  **Attach LoRA:** Injects LoRA adapters into the model's projection layers (query, key, value, output, etc.) with a rank ($r$) of 16. This ensures we only train a tiny fraction of the parameters.
3.  **Prepare Data:** Loads a sample SQL dataset (`b-mc2/sql-create-context`) and formats it using the standard **Alpaca** prompt template (Instruction/Input/Response).
- [Link to Dataset: b-mc2/sql-create-context](https://huggingface.co/datasets/b-mc2/sql-create-context)
4.  **Train:** Sets up the `SFTTrainer` (Supervised Fine-Tuning) with optimized hyperparameters (AdamW 8-bit optimizer, gradient accumulation) and runs a short training session.
5.  **Save:** Saves the trained LoRA adapters and tokenizer to the local `./sql_adapter` directory for future use.

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# --- Configuration ---
max_seq_length = 2048 # Set the maximum number of "tokens" the model can read and generate in a single go
dtype = None # Determine numerical precisio of weights,auto detection based on GPU capabilities
load_in_4bit = True # Enable 4-bit quantization, instead of storing weights as 16-bit

# --- Load Model ---
print("Loading Unsloth Llama 3 Model")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# --- Attach LoRA ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank mentioned above
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Attention and FFN layers
    lora_alpha = 16, # scaling factor for weight update
    lora_dropout = 0,
    bias = "none", # not training bias param, saving on parameters
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- Prepare Data ---
# https://github.com/tatsu-lab/stanford_alpaca
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

dataset = load_dataset("b-mc2/sql-create-context", split = "train[:100]") # Small subset for demo
def formatting_prompts_func(examples):
    instructions = examples["question"]
    inputs       = examples["context"]
    outputs      = examples["answer"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

# --- Train ---
print("Starting Training")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # feed 2 example to GPU at a time, keep VRAM low
        gradient_accumulation_steps = 4, # accumulate result of 4 batches before update model weights
        max_steps = 30, # very short for demo
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), # hardware compatability
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # 8-bit quantization for optimizer states, saving ~75% memory
        output_dir = "outputs",
    ),
)
trainer.train() # Approx 2mins

# --- Save ---
print("Saving LoRA Adapter to ./sql_adapter")
model.save_pretrained("sql_adapter")
tokenizer.save_pretrained("sql_adapter")
print("Done!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Unsloth Llama 3 Model
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3. vLLM: 0.13.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alice-chua (alice-chua-university-of-toronto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.367100
2,2.240900
3,2.509200
4,2.055400
5,2.037800
6,2.054500
7,1.833800
8,1.896000
9,1.750800
10,1.667500


Saving LoRA Adapter to ./sql_adapter
Done!


## Clearing GPU Memory

Training session leave behind "computational leftovers" like the optimizer state, gradients, activations, which persists in the VRAM even after the code stop running.

In [ ]:
import gc
import torch

# Delete variables
try:
    del model
    del tokenizer
    del trainer
except:
    pass

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()

# Double check memory
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Memory cleared. Ready for vLLM.")

GPU Memory Allocated: 2.11 GB
Memory cleared. Ready for vLLM.


In [ ]:
def trace_gpu_memory():
    # Force garbage collection first to clear dead variables
    gc.collect()
    torch.cuda.empty_cache()

    # List to store (size_in_mb, shape, object_type)
    tensors = []

    # Iterate over all objects in memory
    for obj in gc.get_objects():
        try:
            # Check if it's a Tensor or a Parameter on CUDA
            if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
                if obj.is_cuda:
                    size_mb = (obj.element_size() * obj.nelement()) / (1024 ** 2)
                    tensors.append((size_mb, list(obj.size()), type(obj)))
        except:
            pass # Ignore objects that fail access

    # Sort by size (largest first)
    tensors.sort(key=lambda x: x[0], reverse=True)

    # Print Top 10
    print(f"{'SIZE (MB)':<12} | {'SHAPE':<25} | {'TYPE'}")
    print("-" * 60)
    total_tracked = 0
    for size, shape, type_ in tensors[:10]:
        total_tracked += size
        print(f"{size:<12.2f} | {str(shape):<25} | {type_.__name__}")

    print("-" * 60)
    print(f"Total explicitly tracked tensors: {total_tracked:.2f} MB")
    print(f"Total PyTorch Allocated:          {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

trace_gpu_memory()

WARNING 01-14 04:05:19 [interface.py:589] Current platform cuda does not have 'data' attribute.
WARNING 01-14 04:05:19 [interface.py:589] Current platform cuda does not have 'data' attribute.
SIZE (MB)    | SHAPE                     | TYPE
------------------------------------------------------------
2004.00      | [4096, 128256]            | FakeTensor
2004.00      | [128256, 4096]            | FakeTensor
1002.00      | [4096, 128256]            | FakeTensor
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [128256, 4096]            | Parameter
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [4096, 128256]            | FakeTensor
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [128256, 4096]            | Tensor
------------------------------------------------------------
Total explicitly tracked tensors: 12024.00 MB
Total PyTorch Allocated:          2161.94 MB


## Training a second LoRA Adaptor

This section demonstrates how to train a **second, distinct adapter** on the same base model. While the first adapter was for SQL generation, this one fine-tunes the model to mimic **Shakespearean writing**.

### Key Steps:
1.  **Reload Base Model:** We re-initialize the Llama 3 model. This is necessary to attach a fresh, blank set of LoRA adapters (starting from scratch rather than continuing from the SQL training).
2.  **Load Creative Dataset:** We load the `Trelis/tiny-shakespeare` dataset, which contains lines from Shakespeare's plays.
    * [Link to Dataset: Trelis/tiny-shakespeare](https://huggingface.co/datasets/Trelis/tiny-shakespeare)
3.  **Format Data:** We apply a custom formatting function to prompt the model with a specific style trigger: `### Style: Shakespeare`.
4.  **Train:** We run the `SFTTrainer` again with memory-optimized settings (Batch size 2, Gradient Accumulation 4) for 30 steps.
5.  **Save to New Directory:** The trained weights are saved to a separate folder (`./creative_adapter`), ensuring they don't overwrite the previous SQL adapter.
6.  **Final Cleanup:** The script explicitly deletes the model and clears GPU cache (`gc.collect()`, `torch.cuda.empty_cache()`) to free up resources for inference or further tasks.

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# --- Reload Base Model ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    random_state = 3407,
)

# --- Load Creative Dataset ---
dataset = load_dataset("Trelis/tiny-shakespeare", split="train[:100]")

# Simple formatting for raw text to instruction style
def formatting_func(examples):
    text = examples["Text"]
    # if "Text" not in examples and "text" in examples:
    #      text = examples["text"]
    return { "text" : [f"### Style: Shakespeare\n### Content: {t}" for t in text] }

dataset = dataset.map(formatting_func, batched = True)

# --- Train ---
print("Starting Training")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_creative",
    ),
)
trainer.train()

# --- Save to New Directory ---
print("Saving Creative Adapter to ./creative_adapter...")
model.save_pretrained("creative_adapter")
tokenizer.save_pretrained("creative_adapter")

# --- Final Cleanup ---
del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3. vLLM: 0.13.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


README.md:   0%|          | 0.00/497 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/472 [00:00<?, ? examples/s]

...

🎭 Training Creative Adapter...


Map:   0%|          | 0/472 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 472 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,1.187500
2,1.305100
3,1.274000
4,1.060300
5,1.140900
6,1.143300
7,1.123900
8,1.127400
9,1.000000
10,1.054800


💾 Saving Creative Adapter to ./creative_adapter...
✅ Creative Adapter Saved & Memory Cleared.


## vLLM Server

vLLM is a high-performance, open-source engine designed to serve Large Language Models (LLMs) like Llama 3, Mixtral, and Qwen with extremely high throughput and low latency.

It turns a heavy AI model file (weights) into a fast, web-accessible API server that many users can talk to simultaneously.

### Why is it so fast? (PagedAttention)
Traditional servers waste 60-80% of GPU memory due to fragmentation. vLLM uses **PagedAttention**, an algorithm inspired by operating system virtual memory.
* It breaks memory into non-contiguous "pages."
* It allows the server to batch many more users simultaneously.
* **Result:** 14x–24x higher throughput than standard HuggingFace Transformers.

### Multi-LoRA Support
In this notebook, we are using vLLM's **Multi-LoRA** feature.
* **One Base Model:** We load the heavy Llama-3-8B model once.
* **Multiple Adapters:** We can serve both the `sql_adapter` and `creative_adapter` on the same server.
* **Dynamic Switching:** The server swaps the tiny adapter weights on-the-fly depending on which `model` name you request in the API.

### OpenAI Compatible API
vLLM provides an endpoint that looks exactly like OpenAI's. You can use standard libraries to interact with it:

```python
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# Request the SQL Adapter
response = client.chat.completions.create(
    model="sql-model",
    messages=[...]
)

In [6]:
# Find and kill any process using the GPU
!fuser -k -v /dev/nvidia0

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root      27132 F...m vllm
                     root      27357 F...m VLLM::EngineCor


In [8]:
# Check if any processes are running, because we need min 12GB RAM to run the server
!nvidia-smi

Wed Jan 14 04:18:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
import subprocess

vllm_process = subprocess.Popen(
        [
            "vllm", "serve",
            "unsloth/llama-3-8b-bnb-4bit",  # The shared Base Model
            "--quantization", "bitsandbytes",
            "--load-format", "bitsandbytes",
            "--trust-remote-code",
            "--dtype", "half",
            "--enable-lora", # Initialize LoRA management system

            # Syntax: name_in_api=path_to_folder
            "--lora-modules",
            "sql-model=./sql_adapter",
            "shakespeare-model=./creative_adapter",

            "--max-loras", "2",           # Allow switching between 2 adapters
            "--max-model-len", "2048",    # Must fit within memory constraints
            "--gpu-memory-utilization", "0.8", # Reduce base on memory utilization
            "--port", "8000",
            "--enforce-eager",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        start_new_session=True
    )


print(f"vLLM Multi-LoRA Server started with PID: {vllm_process.pid}")

✅ vLLM Multi-LoRA Server started with PID: 29143
Endpoints available:
  1. 'sql-model'
  2. 'shakespeare-model'
  3. 'unsloth/llama-3-8b-bnb-4bit' (The base model)


In [10]:
import requests
import time
from typing import Tuple
import sys

def check_vllm_status(url: str = "http://localhost:8000/health") -> bool:
    """Check if VLLM server is running and healthy."""
    try:
        response = requests.get(url)
        return response.status_code == 200
    except requests.exceptions.ConnectionError:
        return False

def monitor_vllm_process(vllm_process: subprocess.Popen, check_interval: int = 5) -> Tuple[bool, str, str]:
    """
    Monitor VLLM process and return status, stdout, and stderr.
    Returns: (success, stdout, stderr)
    """
    print("Starting VLLM server monitoring...")

    while vllm_process.poll() is None:  # While process is still running
        if check_vllm_status():
            print("✓ VLLM server is up and running!")
            return True, "", ""

        print("Waiting for VLLM server to start...")
        time.sleep(check_interval)

        # Check if there's any output to display
        if vllm_process.stdout.readable():
            stdout = vllm_process.stdout.read1().decode('utf-8')
            if stdout:
                print("STDOUT:", stdout)

        if vllm_process.stderr.readable():
            stderr = vllm_process.stderr.read1().decode('utf-8')
            if stderr:
                print("STDERR:", stderr)

    # If we get here, the process has ended
    stdout, stderr = vllm_process.communicate()
    return False, stdout.decode('utf-8'), stderr.decode('utf-8')

In [11]:
try:
    success, stdout, stderr = monitor_vllm_process(vllm_process)

    if not success:
        print("\n❌ VLLM server failed to start!")
        print("\nFull STDOUT:", stdout)
        print("\nFull STDERR:", stderr)
        sys.exit(1)

except KeyboardInterrupt:
    print("\n⚠️ Monitoring interrupted by user")
    # # This should just exited the process of probing, not the vllm, if you want it then you could uncomment this.
    # vllm_process.terminate()
    # try:
    #     vllm_process.wait(timeout=5)
    # except subprocess.TimeoutExpired:
    #     vllm_process.kill()

    stdout, stderr = vllm_process.communicate()
    if stdout: print("\nFinal STDOUT:", stdout.decode('utf-8'))
    if stderr: print("\nFinal STDERR:", stderr.decode('utf-8'))
    sys.exit(0)

Starting VLLM server monitoring...
Waiting for VLLM server to start...
STDOUT: (APIServer pid=29143) INFO 01-14 04:19:07 [api_server.py:1351] vLLM API server version 0.13.0

STDERR: 2026-01-14 04:18:57.118175: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768364337.138712   29143 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768364337.144829   29143 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768364337.161022   29143 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768364337.161041   29143 computation_placer.cc:177] computation pla

## Querying the server

In [14]:
import time
import requests
from openai import OpenAI

# --- Wait For Server Initializaion ---
def wait_for_server(url="http://localhost:8000/v1/models"):
    print("Waiting for vLLM to load")
    start = time.time()
    while True:
        try:
            response = requests.get(url)
            if response.status_code == 200:
                print(f"Server is ready! (Took {time.time() - start:.1f}s)")
                return
        except requests.exceptions.ConnectionError:
            pass

        # Check if process died
        if vllm_process.poll() is not None:
            print("Server process died unexpectedly!")
            # Read error logs from the pipe if you used PIPE (warning: this can hang if buffer is full)
            out, err = vllm_process.communicate()
            print(err.decode())
            break

        time.sleep(5)

# Wait before connecting
wait_for_server()

# --- Initialize Client ---
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# --- Test SQL LoRA ---
print("\n--- Testing SQL LoRA ---")

prompt_sql = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Get all users from Canada.

### Input:


### Response:
"""

try:
    response = client.completions.create(
        model="sql-model",       # Call the LoRA
        prompt=prompt_sql,       # Send the raw training format
        max_tokens=100,
        temperature=0.1,
        stop=["###"]             # Stop it from hallucinating new instructions
    )
    print(f"SQL Output:\n{response.choices[0].text.strip()}")
except Exception as e:
    print(f"Error: {e}")

# --- Test Shakespeare LoRA ---
print("\n--- Testing Shakespeare LoRA ---")
prompt_shake = "### Style: Shakespeare\n### Content: Explain the concept of love."

try:
    response = client.completions.create(
        model="shakespeare-model",
        prompt=prompt_shake,
        max_tokens=100,
        temperature=0.7
    )
    print(f"Shakespeare Output:\n{response.choices[0].text.strip()}")
except Exception as e:
    print(f"Error: {e}")

⏳ Waiting for vLLM to load (this can take 2 mins)...
✅ Server is ready! (Took 0.0s)

--- Testing SQL LoRA ---
SQL Output:
SELECT * FROM users WHERE country = "Canada";

--- Testing Shakespeare LoRA ---
Shakespeare Output:
How does it change throughout the play? Do you think love is the main theme of the play? Why or why not?

## Introduction
### Love is a very complex and difficult to understand feeling. There are many different types of love and each one can have its own unique qualities. It can be a very powerful and intense feeling, or it can be a gentle and comforting feeling. Love can make people feel happy and fulfilled, or it can make them feel sad and empty. It can be a


In [16]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from openai import OpenAI
import time

# --- Initialize Client ---
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# --- Formatter ---
def format_prompt(model_key, user_input):
    if model_key == "sql-model":
        return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{user_input}

### Input:


### Response:
"""
    elif model_key == "shakespeare-model":
        return f"### Style: Shakespeare\n### Content: {user_input}"
    else:
        return user_input

# --- Create Interactive Widget to test vLLM Server---
style = {'description_width': 'initial'}

header = widgets.HTML("<h2>🧪 LoRA Adapter Playground</h2>")

model_dropdown = widgets.Dropdown(
    options=[
        ('📊 SQL Generator', 'sql-model'),
        ('🎭 Shakespeare', 'shakespeare-model'),
        ('🧠 Base Model (Raw)', 'unsloth/llama-3-8b-bnb-4bit')
    ],
    value='sql-model',
    description='<b>Select Adapter:</b>',
    style=style,
    layout=widgets.Layout(width='50%')
)

input_box = widgets.Textarea(
    value='',
    placeholder='Type your instruction here...',
    description='<b>Your Prompt:</b>',
    style=style,
    layout=widgets.Layout(width='80%', height='100px')
)

# Button with initial state
send_btn = widgets.Button(
    description="Generate Response",
    button_style='primary',
    icon='bolt',
    layout=widgets.Layout(width='20%')
)

out = widgets.Output(layout={'border': '1px solid #444', 'padding': '15px', 'margin_top': '10px'})

def on_click(b):
    prompt_text = input_box.value
    if not prompt_text:
        with out:
            clear_output()
            print("⚠️ Please enter a prompt first.")
        return

    # --- UI STATE: BUSY ---
    # Disable button and change text to show we are working
    send_btn.disabled = True
    send_btn.description = "Running..."
    send_btn.button_style = 'warning' # Turns Orange

    with out:
        clear_output()
        # Show a loading message
        print(f"⏳ Sending query to {model_dropdown.label}...")
        print("   (This might take 5-10 seconds depending on GPU load)")

    try:
        # Prepare Prompt
        full_prompt = format_prompt(model_dropdown.value, prompt_text)

        # API Call
        start_time = time.time()
        response = client.completions.create(
            model=model_dropdown.value,
            prompt=full_prompt,
            max_tokens=200,
            temperature=0.1 if "sql" in model_dropdown.value else 0.7,
            stop=["###"]
        )
        duration = time.time() - start_time

        # --- UI STATE: SUCCESS ---
        with out:
            clear_output()
            print(f"✅ Complete ({duration:.2f}s)\n")
            print("✨ RESPONSE:")
            print("-" * 60)
            print(response.choices[0].text.strip())
            print("-" * 60)

    except Exception as e:
        # --- UI STATE: ERROR ---
        with out:
            print(f"\n❌ Error: {e}")

    finally:
        # --- UI STATE: RESET ---
        # Always re-enable the button, even if it crashed
        send_btn.disabled = False
        send_btn.description = "Generate Response"
        send_btn.button_style = 'primary' # Back to Blue

send_btn.on_click(on_click)

display(header, model_dropdown, input_box, send_btn, out)

HTML(value='<h2>🧪 LoRA Adapter Playground</h2>')

Dropdown(description='<b>Select Adapter:</b>', layout=Layout(width='50%'), options=(('📊 SQL Generator', 'sql-m…

Textarea(value='', description='<b>Your Prompt:</b>', layout=Layout(height='100px', width='80%'), placeholder=…

Button(button_style='primary', description='Generate Response', icon='bolt', layout=Layout(width='20%'), style…

Output(layout=Layout(border='1px solid #444', padding='15px'))

## Final Cleanup

In [17]:
# To terminate (stop) the vLLM server process that is running in the background
!pkill vllm

🛑 Server stopped.
